# Torch-version check for the round-3 PACS test (Colab T4)

**Why:** in the round-3 primary test, every M2 run used torch 2.11.0, but 15 of the 20 M2-CL runs used 2.14.1 (Colab updated mid-way). This check measures how much the torch version alone moves a run. It retrains the four seed-0 M2 runs of the primary test (lr 1e-3, one per PACS fold):
- **Session 1:** torch **2.14.1**, the version the M2-CL runs used.
- **Session 2:** torch **2.11.0**, the same version as the originals, which measures ordinary rerun noise.

The settings, the comparison and how it is read are fixed in advance in `scripts/analyze_torchcheck.py` and `docs/AUDIT.md` F28. Never change a training setting here.

**Cost:** 4 runs per session, about 30 min each on a T4, so about 2 h per session. The two sessions can run on two accounts at the same time.

**Before you start**, the top level of My Drive needs:
- `M2CL_code.zip` (the round-3 one from `colab_upload/`);
- `PACS.zip`.

**Each session:**
1. Runtime > Disconnect and delete runtime (a fresh machine).
2. Runtime > Change runtime type > **T4 GPU**.
3. Set `TORCH_VERSION` in cell 1.
4. Run the cells in order. Cell 1 installs the exact torch version; run it **before anything else imports torch**.

If Colab stops, run all cells again with the same `TORCH_VERSION`: each run resumes from its last saved epoch.

In [ ]:
# 1. Install the exact torch version for this session. Session 1: '2.14.1'. Session 2: '2.11.0'.
TORCH_VERSION = '2.14.1'

assert TORCH_VERSION in ('2.14.1', '2.11.0'), 'TORCH_VERSION must be 2.14.1 or 2.11.0'
import subprocess, sys
!pip -q install "torch=={TORCH_VERSION}" torchvision --index-url https://download.pytorch.org/whl/cu130
!pip -q install pyyaml tabulate
found = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'],
                       capture_output=True, text=True).stdout.strip()
print('torch installed:', found)
assert found.startswith(TORCH_VERSION), (f'torch {found} is installed, not {TORCH_VERSION}. '
                                         'Disconnect and delete the runtime, then run this cell first.')

In [ ]:
# 2. Setup: Drive, code, PACS, and the GPU check
from google.colab import drive
drive.mount('/content/drive')
DRIVE, OUT = '/content/drive/MyDrive', '/content/drive/MyDrive/m2cl_v2'
CODE_ZIP, PACS_ZIP = f'{DRIVE}/M2CL_code.zip', f'{DRIVE}/PACS.zip'

import os, shutil, zipfile, json, platform, torch
assert torch.__version__.startswith(TORCH_VERSION), f'torch {torch.__version__} is loaded: restart the runtime and run cell 1 first.'
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU.'
GPU = torch.cuda.get_device_name(0)
assert 'T4' in GPU, f'This session got a {GPU}. The original runs used a T4: disconnect and get a T4.'
for p in (CODE_ZIP, PACS_ZIP):
    assert os.path.exists(p), f'Not found: {p}. Upload it to the top level of My Drive.'

shutil.rmtree('/content/repo', ignore_errors=True)
zipfile.ZipFile(CODE_ZIP).extractall('/content/repo')
os.chdir('/content/repo')
assert os.path.exists('configs/round3_m2.yaml'), 'This M2CL_code.zip is older than round 3: upload the one from colab_upload/.'

if not os.path.exists('/content/PACS_raw'):
    print('Extracting PACS...')
    zipfile.ZipFile(PACS_ZIP).extractall('/content/PACS_raw')
def is_rgb(root):
    folder = os.path.join(root, 'art_painting', 'dog')
    return os.path.isdir(folder) and not any(f.startswith('dct_') for f in os.listdir(folder))
roots = [r for r, d, _ in os.walk('/content/PACS_raw') if {'art_painting', 'cartoon', 'photo', 'sketch'} <= set(d) and is_rgb(r)]
assert roots, 'No RGB copy of PACS in PACS.zip.'
DATA = roots[0]
n = sum(len(f) for _, _, f in os.walk(DATA))
assert n == 9991, f'Expected 9,991 PACS images, found {n}.'

RUN_DIR = f'{OUT}/torchcheck/torch_{TORCH_VERSION}'
os.makedirs(RUN_DIR, exist_ok=True)
env = {'torch': torch.__version__, 'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(), 'gpu': GPU,
       'python': platform.python_version()}
import datetime
stamp = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
json.dump(env, open(f'{RUN_DIR}/env_{stamp}.json', 'w'), indent=1)
print(env, '| PACS', DATA, '| output', RUN_DIR)

In [ ]:
# 3. The four seed-0 M2 runs (round-3 M2 config, unchanged). About 2 h; resumes after a stop.
!python scripts/train_lodo.py --config configs/round3_m2.yaml --seeds 0 --data_root "$DATA" --out_dir "$RUN_DIR"

In [ ]:
# 4. Status
for t in ('art_painting', 'cartoon', 'photo', 'sketch'):
    p = f'{RUN_DIR}/m2_6tap_{t}_seed0/result.json'
    r = json.load(open(p)) if os.path.exists(p) else {}
    print(f"{t:13s}", 'finished' if r.get('status') == 'complete' else 'not finished',
          f"test {r['test_acc']:.2f} (epoch {r['best_epoch']}, torch {r['torch']})" if r.get('status') == 'complete' else '')

## When both sessions are done
Download `My Drive/m2cl_v2/torchcheck` (both `torch_2.14.1` and `torch_2.11.0`) as a zip, put it in the laptop's `Downloads` folder, and tell Claude. The laptop then runs `scripts/analyze_torchcheck.py` against the originals in archive zip 06. Also say whether any run printed a `[resume]` line.